# Algorithm Experiments — Territory Optimizer

Test different solver configurations head-to-head:
1. **Alpha sweep** — compatibility vs distance weight trade-off
2. **Lambda sweep** — disruption penalty (retention bonus)
3. **Cluster ratio sweep** — greedy solver vs CP-SAT fallback
4. **Travel radius sweep** — constraint tightness
5. **Best config** — run the winning combination

Results are collected into a DataFrame and plotted for comparison.

## 0. Setup

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(message)s')
for lg in ['pipeline', 'optimization', 'features', 'data']:
    logging.getLogger(lg).setLevel(logging.WARNING)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from copy import deepcopy
from pathlib import Path
import time

%matplotlib inline

print('Setup OK')

In [ ]:
# Generate a moderate-size dataset
NUM_DEALERS = 300
NUM_FTCS = 15

data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
for f in data_dir.glob('*.parquet'):
    f.unlink()

from data.generate_synthetic_data import SyntheticDataGenerator
gen = SyntheticDataGenerator(num_dealers=NUM_DEALERS, num_ftcs=NUM_FTCS, seed=42)
raw = gen.generate_all(output_dir='data')

print(f'Dealers:       {len(raw["dealers"])}  ({(raw["dealers"]["dealer_type"]=="static").sum()} static)')
print(f'FTCs:          {len(raw["ftcs"])}')
print(f'Relationships: {len(raw["relationships"])}')

In [ ]:
# Build a helper that runs one config and returns metrics
from config.settings import ConfigManager
from pipeline import OptimizationPipeline

def run_config(overrides: dict) -> dict:
    """Run pipeline with parameter overrides.
    overrides uses dotted keys matching parameters.json sections,
    e.g. {'optimization.alpha_1': 0.7, 'clustering.cluster_ratio': 2.0}
    """
    config = ConfigManager()
    pipeline = OptimizationPipeline(config)
    params = {**overrides, 'solver.time_limit_seconds': 30}
    result = pipeline.run(parameters=params)
    return result

def extract_metrics(r: dict) -> dict:
    """Pull key metrics from a pipeline result dict."""
    return {
        'status': r['status'],
        'objective': r.get('objective_value', 0),
        'changed_dealers': r['changed_dealers'],
        'change_rate': r['change_rate'],
        'ftcs_used': r['ftcs_used'],
        'num_ftcs': r['num_ftcs'],
        'mean_distance_km': r.get('mean_distance_km', 0),
        'median_distance_km': r.get('median_distance_km', 0),
        'solve_time': r['solve_time'],
        'total_pipeline_time': r['total_pipeline_time'],
        'num_clusters': r['num_clusters'],
        'compat_improvement': r.get('compatibility_improvement', 0),
        'distance_reduction': r.get('distance_reduction', 0),
        'workload_balance': r.get('workload_balance_improvement', 0),
        'violations': len(r.get('validation', {}).get('warnings', [])),
    }

print('Helper functions ready')

---
## 1. Alpha Sweep — Compatibility vs Distance

Vary the weight on compatibility (alpha_1) and distance (alpha_2) to see how the trade-off shifts.  
`alpha_1 + alpha_2 + lambda = 1.0` is not enforced by the solver — these are independent weights in the objective:
```
max  alpha_1 * compatibility + alpha_2 * (1 - distance/100) + lambda * kept
```

In [ ]:
alpha_configs = [
    # (label, alpha_1, alpha_2)
    ('Distance-only',    0.0,  0.5),
    ('Compatibility-only', 0.5, 0.0),
    ('Balanced',         0.5,  0.3),
    ('Heavy-compat',     0.7,  0.1),
    ('Heavy-distance',   0.2,  0.5),
    ('Equal-parts',      0.4,  0.4),
]

alpha_results = []
for label, a1, a2 in alpha_configs:
    t0 = time.time()
    r = run_config({'optimization.alpha_1': a1, 'optimization.alpha_2': a2})
    m = extract_metrics(r)
    m['config_label'] = label
    m['alpha_1'] = a1
    m['alpha_2'] = a2
    alpha_results.append(m)
    print(f'{label:20s}  changed={m["changed_dealers"]:3d}  mean_dist={m["mean_distance_km"]:6.1f}km  '
          f'compat_imp={m["compat_improvement"]:+.1%}  obj={m["objective"]:.2f}  ({time.time()-t0:.1f}s)')

df_alpha = pd.DataFrame(alpha_results)
print(f'\nCompleted {len(alpha_configs)} configs')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

x = df_alpha['config_label']

ax = axes[0]
bars = ax.bar(x, df_alpha['mean_distance_km'], color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_ylabel('Mean Distance (km)')
ax.set_title('Distance Impact')
ax.tick_params(axis='x', rotation=25)
for bar, v in zip(bars, df_alpha['mean_distance_km']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{v:.0f}', ha='center', va='bottom', fontsize=9)

ax = axes[1]
bars = ax.bar(x, df_alpha['compat_improvement']*100, color='coral', edgecolor='black', linewidth=0.5)
ax.set_ylabel('Compatibility Improvement (%)')
ax.set_title('Compatibility Impact')
ax.tick_params(axis='x', rotation=25)
for bar, v in zip(bars, df_alpha['compat_improvement']*100):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{v:.1f}', ha='center', va='bottom', fontsize=9)

ax = axes[2]
bars = ax.bar(x, df_alpha['changed_dealers'], color='mediumseagreen', edgecolor='black', linewidth=0.5)
ax.set_ylabel('Dealers Reassigned')
ax.set_title('Disruption (Change Rate)')
ax.tick_params(axis='x', rotation=25)
for bar, v in zip(bars, df_alpha['changed_dealers']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{v}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Alpha Sweep: Compatibility vs Distance Weight', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

---
## 2. Lambda Sweep — Disruption Penalty

`lambda` controls how much the solver penalises changing a dealer's FTC.  
Higher values = fewer reassignments (minimal disruption).  
Lower values = more reshuffling (better objective but harder to implement).

In [ ]:
lambda_configs = [
    ('No retention',    0.0),
    ('Low penalty',     0.2),
    ('Default',         1.0),
    ('High penalty',    2.0),
    ('Very high',       5.0),
    ('Lock everything', 10.0),
]

lambda_results = []
for label, lam in lambda_configs:
    t0 = time.time()
    r = run_config({'optimization.lambda': lam})
    m = extract_metrics(r)
    m['config_label'] = label
    m['lambda'] = lam
    lambda_results.append(m)
    print(f'{label:20s}  changed={m["changed_dealers"]:3d}  mean_dist={m["mean_distance_km"]:6.1f}km  '
          f'compat_imp={m["compat_improvement"]:+.1%}  obj={m["objective"]:.2f}  ({time.time()-t0:.1f}s)')

df_lambda = pd.DataFrame(lambda_results)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

x = df_lambda['config_label']

ax = axes[0]
bars = ax.bar(x, df_lambda['changed_dealers'], color='mediumseagreen', edgecolor='black', linewidth=0.5)
ax.set_ylabel('Dealers Reassigned')
ax.set_title('Disruption vs Lambda')
ax.tick_params(axis='x', rotation=25)
for bar, v in zip(bars, df_lambda['changed_dealers']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{v}', ha='center', va='bottom', fontsize=9)

ax = axes[1]
ax.plot(range(len(df_lambda)), df_lambda['mean_distance_km'], 'o-', color='steelblue', linewidth=2, markersize=8)
ax.set_xticks(range(len(df_lambda)))
ax.set_xticklabels(df_lambda['config_label'], rotation=25)
ax.set_ylabel('Mean Distance (km)')
ax.set_title('Distance vs Lambda')
ax.grid(alpha=0.3)

ax = axes[2]
ax.plot(range(len(df_lambda)), df_lambda['objective'], 'o-', color='coral', linewidth=2, markersize=8)
ax.set_xticks(range(len(df_lambda)))
ax.set_xticklabels(df_lambda['config_label'], rotation=25)
ax.set_ylabel('Objective Score')
ax.set_title('Objective vs Lambda')
ax.grid(alpha=0.3)

plt.suptitle('Lambda Sweep: Disruption Penalty', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

---
## 3. Cluster Ratio Sweep — Greedy vs CP-SAT

`cluster_ratio` controls how many micro-clusters are created.  
- **Low ratio** (`<= 0.5`): fewer clusters, each spans many dealers → the solver **falls back to CP-SAT** (exact but slow on large problems)
- **High ratio** (`>= 1.5`): many small clusters → greedy + local search  
- **cluster_ratio = 0**: no clustering → **forces CP-SAT** path

Compare quality vs solve time between the two solver strategies.

In [ ]:
ratio_configs = [
    ('CP-SAT (ratio=0.0)',  0.0),
    ('CP-SAT (ratio=0.3)',  0.3),
    ('Hybrid (ratio=0.8)',  0.8),
    ('Greedy (ratio=1.0)',  1.0),
    ('Greedy (ratio=1.5)',  1.5),
    ('Greedy (ratio=3.0)',  3.0),
]

ratio_results = []
for label, ratio in ratio_configs:
    t0 = time.time()
    r = run_config({'clustering.cluster_ratio': ratio})
    m = extract_metrics(r)
    m['config_label'] = label
    m['cluster_ratio'] = ratio
    ratio_results.append(m)
    algo = 'CP-SAT' if m['num_clusters'] == 0 else 'Greedy'
    print(f'{label:25s}  algo={algo:6s}  clusters={m["num_clusters"]:2d}  '
          f'changed={m["changed_dealers"]:3d}  mean_dist={m["mean_distance_km"]:6.1f}km  '
          f'solve={m["solve_time"]:.2f}s  obj={m["objective"]:.2f}')

df_ratio = pd.DataFrame(ratio_results)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ax = axes[0]
colors = ['#e74c3c' if 'CP-SAT' in l else '#3498db' for l in df_ratio['config_label']]
bars = ax.bar(df_ratio['config_label'], df_ratio['solve_time'], color=colors, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Solve Time (s)')
ax.set_title('Solver Speed')
ax.tick_params(axis='x', rotation=25)
for bar, v in zip(bars, df_ratio['solve_time']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05, f'{v:.2f}s', ha='center', va='bottom', fontsize=8)

# Add a legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', label='CP-SAT'), Patch(facecolor='#3498db', label='Greedy')]
ax.legend(handles=legend_elements, fontsize=9)

ax = axes[1]
bars = ax.bar(df_ratio['config_label'], df_ratio['mean_distance_km'], color=colors, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Mean Distance (km)')
ax.set_title('Distance Quality')
ax.tick_params(axis='x', rotation=25)
for bar, v in zip(bars, df_ratio['mean_distance_km']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{v:.0f}', ha='center', va='bottom', fontsize=9)

ax = axes[2]
bars = ax.bar(df_ratio['config_label'], df_ratio['objective'], color=colors, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Objective Score')
ax.set_title('Solution Quality')
ax.tick_params(axis='x', rotation=25)
for bar, v in zip(bars, df_ratio['objective']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1, f'{v:.2f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Cluster Ratio Sweep: Greedy vs CP-SAT', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

---
## 4. Travel Radius Sweep

`max_travel_radius_km` controls the hard constraint on dealer-FTC distance.  
- **Tight** (25km): very restricted, may force many violations or infeasible solutions  
- **Loose** (200km): almost unconstrained, but dealers may end up very far

In [ ]:
radius_configs = [
    ('Very tight (25km)',  25),
    ('Tight (50km)',       50),
    ('Moderate (75km)',    75),
    ('Default (100km)',   100),
    ('Loose (150km)',     150),
    ('Very loose (200km)', 200),
]

radius_results = []
for label, radius in radius_configs:
    t0 = time.time()
    r = run_config({'constraints.max_travel_radius_km': radius})
    m = extract_metrics(r)
    m['config_label'] = label
    m['radius_km'] = radius
    radius_results.append(m)
    violations = len(r.get('validation', {}).get('warnings', []))
    print(f'{label:25s}  changed={m["changed_dealers"]:3d}  mean_dist={m["mean_distance_km"]:6.1f}km  '
          f'violations={violations:3d}  obj={m["objective"]:.2f}  ({time.time()-t0:.1f}s)')

df_radius = pd.DataFrame(radius_results)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

x = df_radius['config_label']

ax = axes[0]
bars = ax.bar(x, df_radius['mean_distance_km'], color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_ylabel('Mean Distance (km)')
ax.set_title('Actual Distance vs Radius')
ax.tick_params(axis='x', rotation=25)
for bar, v in zip(bars, df_radius['mean_distance_km']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{v:.0f}', ha='center', va='bottom', fontsize=9)

ax = axes[1]
bars = ax.bar(x, df_radius['violations'], color='tomato', edgecolor='black', linewidth=0.5)
ax.set_ylabel('Constraint Violations')
ax.set_title('Violations vs Radius')
ax.tick_params(axis='x', rotation=25)
for bar, v in zip(bars, df_radius['violations']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, f'{v}', ha='center', va='bottom', fontsize=9)

ax = axes[2]
bars = ax.bar(x, df_radius['solve_time'], color='mediumpurple', edgecolor='black', linewidth=0.5)
ax.set_ylabel('Solve Time (s)')
ax.set_title('Solve Time vs Radius')
ax.tick_params(axis='x', rotation=25)
for bar, v in zip(bars, df_radius['solve_time']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{v:.2f}s', ha='center', va='bottom', fontsize=8)

plt.suptitle('Travel Radius Sweep', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

---
## 5. Summary Comparison

All experiments in one table, sorted by objective score.

In [ ]:
all_runs = []
for d in alpha_results:
    all_runs.append({'experiment': 'Alpha', 'config': d['config_label'],
                     'changed': d['changed_dealers'], 'mean_dist': d['mean_distance_km'],
                     'compat_imp': d['compat_improvement'], 'objective': d['objective'],
                     'solve_time': d['solve_time'], 'violations': d['violations']})
for d in lambda_results:
    all_runs.append({'experiment': 'Lambda', 'config': d['config_label'],
                     'changed': d['changed_dealers'], 'mean_dist': d['mean_distance_km'],
                     'compat_imp': d['compat_improvement'], 'objective': d['objective'],
                     'solve_time': d['solve_time'], 'violations': d['violations']})
for d in ratio_results:
    all_runs.append({'experiment': 'ClusterRatio', 'config': d['config_label'],
                     'changed': d['changed_dealers'], 'mean_dist': d['mean_distance_km'],
                     'compat_imp': d['compat_improvement'], 'objective': d['objective'],
                     'solve_time': d['solve_time'], 'violations': d['violations']})
for d in radius_results:
    all_runs.append({'experiment': 'Radius', 'config': d['config_label'],
                     'changed': d['changed_dealers'], 'mean_dist': d['mean_distance_km'],
                     'compat_imp': d['compat_improvement'], 'objective': d['objective'],
                     'solve_time': d['solve_time'], 'violations': d['violations']})

df_all = pd.DataFrame(all_runs).sort_values('objective', ascending=False).reset_index(drop=True)
df_all.index = df_all.index + 1
df_all.index.name = 'Rank'
df_all

In [ ]:
# Best vs worst across each metric
print('=' * 72)
print('BEST  objective :', df_all.iloc[0]['config'], f'({df_all.iloc[0]["objective"]:.2f})')
print('BEST  mean_dist :', df_all.loc[df_all['mean_dist'].idxmin(), 'config'],
      f'({df_all["mean_dist"].min():.1f}km)')
print('BEST  compat_imp:', df_all.loc[df_all['compat_imp'].idxmax(), 'config'],
      f'({df_all["compat_imp"].max():+.1%})')
print('BEST  solve_time:', df_all.loc[df_all['solve_time'].idxmin(), 'config'],
      f'({df_all["solve_time"].min():.2f}s)')
print('BEST  low_change:', df_all.loc[df_all['changed'].idxmin(), 'config'],
      f'({df_all["changed"].min()} dealers)')
print('=' * 72)

---
## 6. Run the Best Configuration

Take the highest-objective config from the table above and run it in detail with visualisation.

In [ ]:
best_row = df_all.iloc[0]
best_experiment = best_row['experiment']
best_config = best_row['config']

print(f'Best overall: {best_config} (experiment: {best_experiment})')
print()

# Map config label back to actual parameters
params = {'solver.time_limit_seconds': 30}

if best_experiment == 'Alpha':
    # Find matching alpha config
    for label, a1, a2 in alpha_configs:
        if label == best_config:
            params['optimization.alpha_1'] = a1
            params['optimization.alpha_2'] = a2
            break
elif best_experiment == 'Lambda':
    for label, lam in lambda_configs:
        if label == best_config:
            params['optimization.lambda'] = lam
            break
elif best_experiment == 'ClusterRatio':
    for label, ratio in ratio_configs:
        if label == best_config:
            params['clustering.cluster_ratio'] = ratio
            break
elif best_experiment == 'Radius':
    for label, radius in radius_configs:
        if label == best_config:
            params['constraints.max_travel_radius_km'] = radius
            break

print('Parameters:', params)
print()

config = ConfigManager()
pipeline = OptimizationPipeline(config)
result = pipeline.run(parameters=params)

print(f'Status:       {result["status"]}')
print(f'Pipeline:     {result["total_pipeline_time"]:.2f}s')
print(f'Solve time:   {result["solve_time"]:.2f}s')
print(f'Active FTCs:  {result["ftcs_used"]}/{result["num_ftcs"]}')
print(f'Reassigned:   {result["changed_dealers"]}/{result["num_dealers"]} ({result["change_rate"]*100:.0f}%)')
print(f'Clusters:     {result["num_clusters"]}')

In [ ]:
# Full report
from analysis.reporting import ReportGenerator
report = ReportGenerator()
print(report.generate_summary_report(result))

### Before / After Map

In [ ]:
from data.loader import DataLoader
from data.processor import DataProcessor
from optimization.model import TerritoryModel

loader = DataLoader()
data_raw = {
    'dealers': loader.load_dealers(),
    'ftcs': loader.load_ftcs(),
    'relationships': loader.load_relationships(),
    'proximity': loader.load_proximity(),
}
dealers = data_raw['dealers']
ftcs = data_raw['ftcs']

proc = DataProcessor()
processed = proc.process_all_data(data_raw)

clusterer = SpatialClusterer(config.get_section('clustering'))
processed['cluster_labels'] = clusterer.fit_predict(processed['dealers'], len(processed['ftc_ids']))

model = TerritoryModel(pipeline._build_run_config(params))
features = pipeline._engineer_features(processed)
solution = model.build_and_solve(features)

new_assign = solution['assignments']
old_assign = solution['current_assignment']
dealer_ids = processed['dealer_ids']
ftc_ids = processed['ftc_ids']

ftc_ids_arr = np.array(ftc_ids)
new_ftc_of_dealer = ftc_ids_arr[new_assign.argmax(axis=1)]
old_ftc_of_dealer = ftc_ids_arr[old_assign.argmax(axis=1)]
changed = new_ftc_of_dealer != old_ftc_of_dealer

print(f'Reassigned: {changed.sum()} / {len(changed)} dealers')
print()
for i, did in enumerate(dealer_ids[:30]):
    if changed[i]:
        print(f'  {did}: {old_ftc_of_dealer[i]} -> {new_ftc_of_dealer[i]}')

In [ ]:
palette = plt.cm.Set1(np.linspace(0, 1, len(ftc_ids)))
ftc_color = dict(zip(ftc_ids, palette))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for title, ax, assign in [('Before', ax1, old_ftc_of_dealer), ('After', ax2, new_ftc_of_dealer)]:
    for i, row in dealers.iterrows():
        c = ftc_color[assign[i]]
        ax.scatter(row['dealer_longitude'], row['dealer_latitude'],
                   c=[c], s=15, alpha=0.7, edgecolors='black', linewidths=0.2)
    for _, ftc in ftcs.iterrows():
        ax.scatter(ftc['ftc_longitude'], ftc['ftc_latitude'],
                   c='green', s=100, marker='^', edgecolors='black', linewidths=0.5, zorder=5)
        ax.annotate(ftc['ftc_id'], (ftc['ftc_longitude'], ftc['ftc_latitude']),
                    textcoords='offset points', xytext=(0, 10), ha='center', fontsize=7, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='gray', alpha=0.7))
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

handles = [mpatches.Patch(color=ftc_color[f], label=f, linewidth=0.5, edgecolor='black') for f in ftc_ids]
handles.append(plt.Line2D([0], [0], marker='^', color='w', markerfacecolor='green', markersize=10, label='FTC'))
fig.legend(handles=handles, loc='lower center', ncol=len(ftc_ids) + 1, fontsize=8, framealpha=0.9)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

### FTC Load Distribution

In [ ]:
old_counts = pd.Series(old_ftc_of_dealer).value_counts().reindex(ftc_ids, fill_value=0)
new_counts = pd.Series(new_ftc_of_dealer).value_counts().reindex(ftc_ids, fill_value=0)

x = np.arange(len(ftc_ids))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w/2, old_counts.values, w, label='Before', color='lightcoral', edgecolor='black', linewidth=0.5)
ax.bar(x + w/2, new_counts.values, w, label='After', color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(ftc_ids, fontsize=8)
ax.set_ylabel('Dealers'); ax.set_title('Dealers per FTC')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

---
## 7. Custom Experiment — Tweak Your Own Config

Edit the cell below and re-run to try any combination.

In [ ]:
custom_params = {
    'optimization.alpha_1': 0.5,
    'optimization.alpha_2': 0.3,
    'optimization.lambda': 0.1,
    'clustering.cluster_ratio': 1.5,
    'constraints.max_travel_radius_km': 100,
    'solver.time_limit_seconds': 30,
}

r = run_config(custom_params)
m = extract_metrics(r)
print('Results:')
for k, v in m.items():
    print(f'  {k}: {v}')
print()
print(f'Violations: {len(r.get("validation", {}).get("warnings", []))}')
for w in r.get('validation', {}).get('warnings', []):
    print(f'  - {w}')

---
## 8. Appendix: Configuration Reference

| Parameter | Default | Description |
|-----------|---------|-------------|
| `optimization.alpha_1` | 0.5 | Weight on compatibility in the objective |
| `optimization.alpha_2` | 0.3 | Weight on (1 - distance) in the objective |
| `optimization.lambda` | 0.1 | Disruption penalty (retention bonus) |
| `clustering.cluster_ratio` | 1.5 | Ratio of clusters to FTCs |
| `clustering.min_cluster_size` | 3 | Minimum dealers per cluster |
| `clustering.max_cluster_size` | 25 | Maximum dealers per cluster |
| `constraints.max_travel_radius_km` | 50 | Max feasible travel distance (km) |
| `constraints.max_dealers_per_ftc` | 25 | Capacity per FTC |
| `constraints.min_dealers_per_ftc` | 1 | Minimum dealers per FTC (relaxed for small data) |
| `solver.time_limit_seconds` | 300 | CP-SAT wall-clock limit |
| `solver.num_workers` | 4 | CP-SAT parallel threads |

**Solver routing rule:**  
If `cluster_labels` is available (cluster_ratio > 0), the **greedy + local search** solver is used.  
Otherwise, the **OR-Tools CP-SAT** exact solver is used (small problems only — scales poorly).